# Demo: `backtest()` method

This notebook demonstrates the `ForecastingAssistant.backtest()` method for every supported forecaster type.

The backtesting workflow:
1. Profiles the data
2. Generates a plan
3. Creates a cross-validation strategy (`TimeSeriesFold`)
4. Generates and executes backtesting code via `exec()`
5. Returns a `BacktestResult` with metrics, predictions, and the exact code that was run

**Key guarantee:** `result.code` is the exact script that produced `result.metrics` and `result.predictions`.

We show two scenarios for each forecaster:
- **Without exogenous variables**
- **With exogenous variables**

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import warnings

import numpy as np
import pandas as pd
from skforecast.model_selection import TimeSeriesFold

from skforecast_ai import ForecastingAssistant

warnings.filterwarnings("ignore")
assistant = ForecastingAssistant()

/opt/homebrew/Caskroom/miniconda/base/envs/skforecast_ai_py13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Synthetic Data

We create datasets for each forecaster type, with and without exogenous variables.

In [3]:
rng = np.random.default_rng(42)
n = 365
dates = pd.date_range("2022-01-01", periods=n, freq="D")

# Single series target
target = 100 + np.cumsum(rng.normal(0, 1, n)) + 5 * np.sin(2 * np.pi * np.arange(n) / 7)

# Exogenous variables
promo = rng.normal(50, 10, n)
temperature = 15 + 10 * np.sin(2 * np.pi * np.arange(n) / 365) + rng.normal(0, 2, n)

# Single series without exog
df_single = pd.DataFrame({
    "date": dates,
    "sales": target,
})

# Single series with exog
df_single_exog = pd.DataFrame({
    "date": dates,
    "sales": target,
    "promo": promo,
    "temperature": temperature,
})

# Multi-series (long format) without exog
n_multi = 200
dates_multi = pd.date_range("2022-01-01", periods=n_multi, freq="D")
df_multi = pd.DataFrame({
    "date": np.tile(dates_multi, 3),
    "series_id": ["store_A"] * n_multi + ["store_B"] * n_multi + ["store_C"] * n_multi,
    "revenue": np.concatenate([
        100 + np.cumsum(rng.normal(0, 1, n_multi)),
        200 + np.cumsum(rng.normal(0, 1.5, n_multi)),
        150 + np.cumsum(rng.normal(0, 0.8, n_multi)),
    ]),
})

# Multi-series (long format) with exog
df_multi_exog = df_multi.copy()
df_multi_exog["promo"] = rng.normal(50, 10, n_multi * 3)

# Wide format for multivariate without exog
df_wide = pd.DataFrame({
    "date": dates[:n_multi],
    "temperature": 20 + np.cumsum(rng.normal(0, 0.5, n_multi)),
    "humidity": 60 + np.cumsum(rng.normal(0, 0.3, n_multi)),
    "pressure": 1013 + np.cumsum(rng.normal(0, 0.2, n_multi)),
})

print(f"df_single:      {df_single.shape}")
print(f"df_single_exog: {df_single_exog.shape}")
print(f"df_multi:       {df_multi.shape}")
print(f"df_multi_exog:  {df_multi_exog.shape}")
print(f"df_wide:        {df_wide.shape}")

df_single:      (365, 2)
df_single_exog: (365, 4)
df_multi:       (600, 3)
df_multi_exog:  (600, 4)
df_wide:        (200, 4)


---
## 1. ForecasterRecursive (single series)

Recursive multi-step prediction using a single global model.

### Without exogenous variables

In [4]:
# Step-by-step: profile → plan → create_cv → backtest
profile_recursive = assistant.profile(
    data        = df_single,
    target      = "sales",
    date_column = "date",
)

plan_recursive = assistant.plan(
    profile    = profile_recursive,
    steps      = 14,
    forecaster = "ForecasterRecursive",
    estimator  = "LGBMRegressor",
)

cv_recursive, cv_explanation = assistant.create_cv(
    profile = profile_recursive,
    plan    = plan_recursive,
)

result_recursive = assistant.backtest(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    cv          = cv_recursive,
    profile     = profile_recursive,
    plan        = plan_recursive,
)

print("Explanation:")
print(result_recursive.explanation)
print("\nMetrics:")
display(result_recursive.metrics)
print("\nPredictions (first rows):")
display(result_recursive.predictions.head(10))

100%|██████████| 8/8 [00:00<00:00, 14.68it/s]

Explanation:
Using 70% of data (255 observations) for initial training, expanding window, refit every fold, 14-step horizon, 8 folds. Results — mean_absolute_error: 3.1531, mean_squared_error: 13.9058, mean_absolute_scaled_error: 1.0714, mean_absolute_percentage_error: 0.0337.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,3.153083,13.90581,1.071439,0.033666



Predictions (first rows):


,fold,pred
2022-09-13,0,94.014981
2022-09-14,0,90.256913
2022-09-15,0,86.088226
2022-09-16,0,83.615495
2022-09-17,0,88.015413
2022-09-18,0,91.882177
2022-09-19,0,94.837231
2022-09-20,0,93.015496
2022-09-21,0,89.063318
2022-09-22,0,84.460885


In [5]:
# Inspect the generated code
print(result_recursive.code)

import pandas as pd
from lightgbm import LGBMRegressor
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import backtesting_forecaster, TimeSeriesFold
from skforecast.preprocessing import RollingFeatures

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'mean'],
    window_sizes = [7, 91],
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator            = LGBMRegressor(random_state=123, verbose=-1),
    lags                 = [1, 3, 7, 8, 10, 12, 118],
    window_features      = window_features,
    categorical_features = 'auto',
    dropna_from_series   = False,
)

# Cross-validation configuration
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = 255,
    refit              = True,
    fixed_train_size   = False,
)

# Run backtest

### With exogenous variables

In [6]:
result_recursive_exog = assistant.backtest(
    data        = df_single_exog,
    target      = "sales",
    date_column = "date",
    cv          = cv_recursive,
    forecaster  = "ForecasterRecursive",
    estimator   = "LGBMRegressor",
)

print("Explanation:")
print(result_recursive_exog.explanation)
print("\nMetrics:")
display(result_recursive_exog.metrics)
print("\nPredictions (first rows):")
display(result_recursive_exog.predictions.head(10))

100%|██████████| 8/8 [00:00<00:00, 15.86it/s]

Explanation:
Using 70% of data (255 observations) for initial training, expanding window, refit every fold, 14-step horizon, 8 folds. Results — mean_absolute_error: 3.1849, mean_squared_error: 14.4905, mean_absolute_scaled_error: 1.0822, mean_absolute_percentage_error: 0.0340.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,3.184862,14.490499,1.082238,0.034049



Predictions (first rows):


,fold,pred
2022-09-13,0,93.641670
2022-09-14,0,90.002343
2022-09-15,0,85.876926
2022-09-16,0,83.741837
2022-09-17,0,87.837475
2022-09-18,0,91.933170
2022-09-19,0,94.988806
2022-09-20,0,93.758693
2022-09-21,0,89.148648
2022-09-22,0,84.682383


In [7]:
# Inspect the generated code (note exog handling)
print(result_recursive_exog.code)

import pandas as pd
from lightgbm import LGBMRegressor
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import backtesting_forecaster, TimeSeriesFold
from skforecast.preprocessing import RollingFeatures

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'mean'],
    window_sizes = [7, 91],
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator            = LGBMRegressor(random_state=123, verbose=-1),
    lags                 = [1, 3, 7, 8, 10, 12, 118],
    window_features      = window_features,
    categorical_features = 'auto',
    dropna_from_series   = False,
)

# Cross-validation configuration
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = 255,
    refit              = True,
    fixed_train_size   = False,
)

# Run backtest

---
## 2. ForecasterDirect (single series)

Trains one model per step. Better for long horizons or non-stationary DGPs.

### Without exogenous variables

In [8]:
profile_direct = assistant.profile(
    data        = df_single,
    target      = "sales",
    date_column = "date",
)

plan_direct = assistant.plan(
    profile    = profile_direct,
    steps      = 14,
    forecaster = "ForecasterDirect",
    estimator  = "Ridge",
)

cv_direct, _ = assistant.create_cv(
    profile = profile_direct,
    plan    = plan_direct,
)

result_direct = assistant.backtest(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    cv          = cv_direct,
    profile     = profile_direct,
    plan        = plan_direct,
)

print("Explanation:")
print(result_direct.explanation)
print("\nMetrics:")
display(result_direct.metrics)
print("\nPredictions (first rows):")
display(result_direct.predictions.head(10))

100%|██████████| 8/8 [00:00<00:00, 320.88it/s]

Explanation:
Using 70% of data (255 observations) for initial training, expanding window, refit every fold, 14-step horizon, 8 folds. Results — mean_absolute_error: 2.5635, mean_squared_error: 9.1800, mean_absolute_scaled_error: 0.8711, mean_absolute_percentage_error: 0.0274.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,2.563548,9.180011,0.871111,0.027366



Predictions (first rows):


,fold,pred
2022-09-13,0,90.944243
2022-09-14,0,86.693301
2022-09-15,0,84.326810
2022-09-16,0,85.750811
2022-09-17,0,89.857726
2022-09-18,0,93.515195
2022-09-19,0,93.964778
2022-09-20,0,91.137217
2022-09-21,0,86.482605
2022-09-22,0,83.890407


In [9]:
print(result_direct.code)

import pandas as pd
from sklearn.linear_model import Ridge
from skforecast.direct import ForecasterDirect
from skforecast.model_selection import backtesting_forecaster, TimeSeriesFold
from skforecast.preprocessing import RollingFeatures
from sklearn.preprocessing import StandardScaler

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'mean'],
    window_sizes = [7, 91],
)

# Create forecaster
forecaster = ForecasterDirect(
    estimator            = Ridge(),
    steps                = 14,
    lags                 = [1, 3, 7, 8, 10, 12, 118],
    window_features      = window_features,
    transformer_y        = StandardScaler(),
    categorical_features = 'auto',
    dropna_from_series   = False,
)

# Cross-validation configuration
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = 2

### With exogenous variables

In [10]:
result_direct_exog = assistant.backtest(
    data        = df_single_exog,
    target      = "sales",
    date_column = "date",
    cv          = cv_direct,
    forecaster  = "ForecasterDirect",
    estimator   = "Ridge",
)

print("Explanation:")
print(result_direct_exog.explanation)
print("\nMetrics:")
display(result_direct_exog.metrics)
print("\nPredictions (first rows):")
display(result_direct_exog.predictions.head(10))

100%|██████████| 8/8 [00:00<00:00, 245.47it/s]

Explanation:
Using 70% of data (255 observations) for initial training, expanding window, refit every fold, 14-step horizon, 8 folds. Results — mean_absolute_error: 2.2590, mean_squared_error: 7.1757, mean_absolute_scaled_error: 0.7676, mean_absolute_percentage_error: 0.0242.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,2.259014,7.175671,0.767628,0.024193



Predictions (first rows):


,fold,pred
2022-09-13,0,90.943351
2022-09-14,0,86.417217
2022-09-15,0,84.246275
2022-09-16,0,85.469643
2022-09-17,0,89.421329
2022-09-18,0,93.498280
2022-09-19,0,94.009019
2022-09-20,0,91.309149
2022-09-21,0,86.505752
2022-09-22,0,83.662214


In [11]:
print(result_direct_exog.code)

import pandas as pd
from sklearn.linear_model import Ridge
from skforecast.direct import ForecasterDirect
from skforecast.model_selection import backtesting_forecaster, TimeSeriesFold
from skforecast.preprocessing import RollingFeatures
from sklearn.preprocessing import StandardScaler

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'mean'],
    window_sizes = [7, 91],
)

transformer_exog = StandardScaler()

# Create forecaster
forecaster = ForecasterDirect(
    estimator            = Ridge(),
    steps                = 14,
    lags                 = [1, 3, 7, 8, 10, 12, 118],
    window_features      = window_features,
    transformer_y        = StandardScaler(),
    transformer_exog     = transformer_exog,
    categorical_features = 'auto',
    dropna_from_series   = False,
)

# Cross-validation configur

---
## 3. ForecasterRecursiveMultiSeries (multi-series)

Global model trained on all series simultaneously. Ideal when series share common patterns.

### Without exogenous variables

In [12]:
profile_multi = assistant.profile(
    data             = df_multi,
    target           = "revenue",
    date_column      = "date",
    series_id_column = "series_id",
)

plan_multi = assistant.plan(
    profile    = profile_multi,
    steps      = 7,
    forecaster = "ForecasterRecursiveMultiSeries",
    estimator  = "LGBMRegressor",
)

cv_multi, _ = assistant.create_cv(
    profile = profile_multi,
    plan    = plan_multi,
)

result_multi = assistant.backtest(
    data             = df_multi,
    target           = "revenue",
    date_column      = "date",
    series_id_column = "series_id",
    cv               = cv_multi,
    profile          = profile_multi,
    plan             = plan_multi,
)

print("Explanation:")
print(result_multi.explanation)
print("\nMetrics (per series):")
display(result_multi.metrics)
print("\nPredictions (first rows):")
display(result_multi.predictions.head(10))

100%|██████████| 7/7 [00:00<00:00, 29.32it/s]

Explanation:
Using 78% of data (157 observations) for initial training, expanding window, refit every fold, 7-step horizon, 7 folds. Results — mean_absolute_error: 10.1698, mean_squared_error: 388.2984, mean_absolute_scaled_error: 11.3657, mean_absolute_percentage_error: 0.0726.

Metrics (per series):


,levels,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,store_A,9.472984,369.165281,11.919759,0.098726
1,store_B,15.094715,709.710145,12.163524,0.073148
2,store_C,5.941556,86.019869,9.889941,0.045812
3,average,10.169752,388.298432,11.324408,0.072562
4,weighted_average,10.169752,388.298432,11.324408,0.072562
5,pooling,10.169752,388.298432,11.571969,0.072562



Predictions (first rows):


,level,fold,pred
2022-06-07,store_A,0,142.474526
2022-06-07,store_B,0,142.474526
2022-06-07,store_C,0,142.474526
2022-06-08,store_A,0,142.474526
2022-06-08,store_B,0,142.474526
2022-06-08,store_C,0,142.474526
2022-06-09,store_A,0,142.474526
2022-06-09,store_B,0,142.474526
2022-06-09,store_C,0,142.474526
2022-06-10,store_A,0,142.474526


In [13]:
print(result_multi.code)

import pandas as pd
from lightgbm import LGBMRegressor
from skforecast.recursive import ForecasterRecursiveMultiSeries
from skforecast.model_selection import backtesting_forecaster_multiseries, TimeSeriesFold
from skforecast.preprocessing import RollingFeatures, reshape_series_long_to_dict

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.sort_values('date')

# Reshape to dict format (required for backtesting multi-series)
series_dict = reshape_series_long_to_dict(
    data      = data,
    series_id = 'series_id',
    index     = 'date',
    values    = 'revenue',
    freq      = 'D',
)

window_features = RollingFeatures(
    stats        = ['mean', 'mean'],
    window_sizes = [7, 150],
)

# Create forecaster
forecaster = ForecasterRecursiveMultiSeries(
    estimator            = LGBMRegressor(random_state=123, verbose=-1),
    lags                 = [1, 2, 7, 18, 24, 42, 88],
    encoding             = 'ordinal',
    window_features 

### With exogenous variables

In [14]:
result_multi_exog = assistant.backtest(
    data             = df_multi_exog,
    target           = "revenue",
    date_column      = "date",
    series_id_column = "series_id",
    cv               = cv_multi,
    forecaster       = "ForecasterRecursiveMultiSeries",
    estimator        = "LGBMRegressor",
)

print("Explanation:")
print(result_multi_exog.explanation)
print("\nMetrics (per series):")
display(result_multi_exog.metrics)
print("\nPredictions (first rows):")
display(result_multi_exog.predictions.head(10))

100%|██████████| 7/7 [00:00<00:00, 29.71it/s]

Explanation:
Using 78% of data (157 observations) for initial training, expanding window, refit every fold, 7-step horizon, 7 folds. Results — mean_absolute_error: 10.0437, mean_squared_error: 385.8405, mean_absolute_scaled_error: 11.1499, mean_absolute_percentage_error: 0.0713.

Metrics (per series):


,levels,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,store_A,9.279120,368.934262,11.675822,0.096739
1,store_B,15.258092,715.127790,12.295175,0.073931
2,store_C,5.593974,73.459523,9.311378,0.043152
3,average,10.043729,385.840525,11.094125,0.071274
4,weighted_average,10.043729,385.840525,11.094125,0.071274
5,pooling,10.043729,385.840525,11.428570,0.071274



Predictions (first rows):


,level,fold,pred
2022-06-07,store_A,0,142.474526
2022-06-07,store_B,0,142.474526
2022-06-07,store_C,0,142.474526
2022-06-08,store_A,0,142.474526
2022-06-08,store_B,0,142.474526
2022-06-08,store_C,0,142.474526
2022-06-09,store_A,0,142.474526
2022-06-09,store_B,0,142.474526
2022-06-09,store_C,0,142.474526
2022-06-10,store_A,0,142.474526


In [15]:
print(result_multi_exog.code)

import pandas as pd
from lightgbm import LGBMRegressor
from skforecast.recursive import ForecasterRecursiveMultiSeries
from skforecast.model_selection import backtesting_forecaster_multiseries, TimeSeriesFold
from skforecast.preprocessing import RollingFeatures, reshape_series_long_to_dict, reshape_exog_long_to_dict

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.sort_values('date')

# Reshape to dict format (required for backtesting multi-series)
series_dict = reshape_series_long_to_dict(
    data      = data,
    series_id = 'series_id',
    index     = 'date',
    values    = 'revenue',
    freq      = 'D',
)

exog_dict = reshape_exog_long_to_dict(
    data      = data[['series_id', 'date', 'promo']],
    series_id = 'series_id',
    index     = 'date',
    freq      = 'D',
)

window_features = RollingFeatures(
    stats        = ['mean', 'mean'],
    window_sizes = [7, 150],
)

# Create forecaster
forecaster = ForecasterRecursive

---
## 4. ForecasterDirectMultiVariate (multivariate)

Uses all series as features to predict one target series. Direct strategy.

Note: This forecaster uses all columns in `target` as both inputs and outputs — exogenous handling is implicit in the multivariate design.

### Without exogenous variables

In [16]:
profile_multivariate = assistant.profile(
    data        = df_wide,
    target      = ["temperature", "humidity", "pressure"],
    date_column = "date",
)

plan_multivariate = assistant.plan(
    profile    = profile_multivariate,
    steps      = 7,
    forecaster = "ForecasterDirectMultiVariate",
    estimator  = "Ridge",
)

cv_multivariate, _ = assistant.create_cv(
    profile = profile_multivariate,
    plan    = plan_multivariate,
)

result_multivariate = assistant.backtest(
    data        = df_wide,
    target      = ["temperature", "humidity", "pressure"],
    date_column = "date",
    cv          = cv_multivariate,
    profile     = profile_multivariate,
    plan        = plan_multivariate,
)

print("Explanation:")
print(result_multivariate.explanation)
print("\nMetrics (per series):")
display(result_multivariate.metrics)
print("\nPredictions (first rows):")
display(result_multivariate.predictions.head(10))

100%|██████████| 7/7 [00:00<00:00, 157.29it/s]

Explanation:
Using 78% of data (157 observations) for initial training, expanding window, refit every fold, 7-step horizon, 7 folds. Results — mean_absolute_error: 1.6727, mean_squared_error: 4.1825, mean_absolute_scaled_error: 4.8655, mean_absolute_percentage_error: 0.0929.

Metrics (per series):


,levels,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,temperature,1.672739,4.182488,4.865497,0.092895



Predictions (first rows):


,level,fold,pred
2022-06-07,temperature,0,22.150894
2022-06-08,temperature,0,21.962842
2022-06-09,temperature,0,21.733801
2022-06-10,temperature,0,22.118284
2022-06-11,temperature,0,22.527836
2022-06-12,temperature,0,22.260293
2022-06-13,temperature,0,22.058711
2022-06-14,temperature,1,22.422687
2022-06-15,temperature,1,22.003333
2022-06-16,temperature,1,21.702005


In [17]:
print(result_multivariate.code)

import pandas as pd
from sklearn.linear_model import Ridge
from skforecast.direct import ForecasterDirectMultiVariate
from skforecast.model_selection import backtesting_forecaster_multiseries, TimeSeriesFold
from skforecast.preprocessing import RollingFeatures
from sklearn.preprocessing import StandardScaler

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean'],
    window_sizes = [7, 7, 150],
)

# Create forecaster
forecaster = ForecasterDirectMultiVariate(
    estimator          = Ridge(),
    level              = 'temperature',
    steps              = 7,
    lags               = [1, 7, 8, 67],
    window_features    = window_features,
    transformer_series = StandardScaler(),
    dropna_from_series = False,
)

# Cross-validation configuration
cv = TimeSeriesFold(
    steps              = 7,


---
## 5. ForecasterStats (ARIMA)

Statistical model. No external estimator needed. Supports native prediction intervals.

### Without exogenous variables

In [18]:
profile_stats = assistant.profile(
    data        = df_single,
    target      = "sales",
    date_column = "date",
)

plan_stats = assistant.plan(
    profile    = profile_stats,
    steps      = 14,
    forecaster = "ForecasterStats",
    interval   = [10, 90],
)

cv_stats, _ = assistant.create_cv(
    profile = profile_stats,
    plan    = plan_stats,
)

result_stats = assistant.backtest(
    data        = df_single,
    target      = "sales",
    date_column = "date",
    cv          = cv_stats,
    profile     = profile_stats,
    plan        = plan_stats,
)

print("Explanation:")
print(result_stats.explanation)
print("\nMetrics:")
display(result_stats.metrics)
print("\nPredictions with intervals (first rows):")
display(result_stats.predictions.head(10))

100%|██████████| 8/8 [00:09<00:00,  1.17s/it]

Explanation:
Using 70% of data (255 observations) for initial training, expanding window, refit every fold, 14-step horizon, 8 folds. Results — mean_absolute_error: 1.9869, mean_squared_error: 6.3830, mean_absolute_scaled_error: 0.6852, mean_absolute_percentage_error: 0.0213.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,1.986883,6.383001,0.685211,0.02133



Predictions with intervals (first rows):


,fold,pred,lower_bound,upper_bound
2022-09-13,0,91.212669,89.912401,92.512937
2022-09-14,0,85.725797,83.779467,87.672127
2022-09-15,0,81.874398,79.515980,84.232816
2022-09-16,0,83.678542,81.045747,86.311337
2022-09-17,0,87.631625,84.809819,90.453431
2022-09-18,0,90.726901,87.771706,93.682096
2022-09-19,0,91.968411,88.917518,95.019304
2022-09-20,0,90.480083,87.214511,93.745655
2022-09-21,0,84.850853,81.394300,88.307406
2022-09-22,0,81.185698,77.584564,84.786832


In [19]:
print(result_stats.code)

import pandas as pd
from skforecast.stats import Arima
from skforecast.recursive import ForecasterStats
from skforecast.model_selection import backtesting_stats, TimeSeriesFold

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

# Create forecaster (Auto-ARIMA)
forecaster = ForecasterStats(
    estimator = Arima(order=None, seasonal_order=None, m=7),
)

# Cross-validation configuration
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = 255,
    refit              = True,
    fixed_train_size   = False,
)

# Run backtesting
metrics, predictions = backtesting_stats(
    forecaster        = forecaster,
    y                 = data['sales'],
    cv                = cv,
    metric            = ['mean_absolute_error', 'mean_squared_error', 'mean_absolute_scaled_error', 'mean_absolute_percentage_error'],
    interval          = [10, 90],
    freeze_param

### With exogenous variables

In [20]:
result_stats_exog = assistant.backtest(
    data        = df_single_exog,
    target      = "sales",
    date_column = "date",
    cv          = cv_stats,
    forecaster  = "ForecasterStats",
    interval    = [10, 90],
)

print("Explanation:")
print(result_stats_exog.explanation)
print("\nMetrics:")
display(result_stats_exog.metrics)
print("\nPredictions with intervals (first rows):")
display(result_stats_exog.predictions.head(10))

100%|██████████| 8/8 [00:08<00:00,  1.12s/it]

Explanation:
Using 70% of data (255 observations) for initial training, expanding window, refit every fold, 14-step horizon, 8 folds. Results — mean_absolute_error: 2.6827, mean_squared_error: 13.7065, mean_absolute_scaled_error: 0.9252, mean_absolute_percentage_error: 0.0287.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,2.682687,13.706543,0.925171,0.028696



Predictions with intervals (first rows):


,fold,pred,lower_bound,upper_bound
2022-09-13,0,89.616790,88.223232,91.010348
2022-09-14,0,84.372231,82.011129,86.733333
2022-09-15,0,81.831669,78.823313,84.840025
2022-09-16,0,84.314566,81.115346,87.513787
2022-09-17,0,88.627748,85.408359,91.847136
2022-09-18,0,91.380037,88.160649,94.599426
2022-09-19,0,91.715577,88.486479,94.944676
2022-09-20,0,89.252317,85.794630,92.710004
2022-09-21,0,84.943654,81.053395,88.833913
2022-09-22,0,82.972947,78.741521,87.204374


In [21]:
print(result_stats_exog.code)

import pandas as pd
from skforecast.stats import Arima
from skforecast.recursive import ForecasterStats
from skforecast.model_selection import backtesting_stats, TimeSeriesFold

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

# Create forecaster (Auto-ARIMA)
forecaster = ForecasterStats(
    estimator = Arima(order=None, seasonal_order=None, m=7),
)

# Cross-validation configuration
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = 255,
    refit              = True,
    fixed_train_size   = False,
)

# Run backtesting
exog_features = ['promo', 'temperature']

metrics, predictions = backtesting_stats(
    forecaster        = forecaster,
    y                 = data['sales'],
    exog              = data[exog_features],
    cv                = cv,
    metric            = ['mean_absolute_error', 'mean_squared_error', 'mean_absolute_scaled_error'

---
## 6. Shortcut: `backtest()` without pre-computed profile/plan

The `backtest()` method can also be called directly with just data and a `TimeSeriesFold`. It will internally profile and plan before running.

In [22]:
# Create a simple CV manually
cv_simple = TimeSeriesFold(
    steps=14,
    initial_train_size=300,
    refit=False,
    fixed_train_size=False,
)

# Backtest with automatic profiling and planning
result_shortcut = assistant.backtest(
    data        = df_single_exog,
    target      = "sales",
    date_column = "date",
    cv          = cv_simple,
)

print("Auto-selected forecaster:", result_shortcut.plan.forecaster)
print("Auto-selected estimator:", result_shortcut.plan.estimator)
print("Uses exog:", result_shortcut.plan.use_exog)
print("\nExplanation:")
print(result_shortcut.explanation)
print("\nMetrics:")
display(result_shortcut.metrics)

Information of folds
--------------------
Number of observations used for initial training: 300
Number of observations used for backtesting: 65
    Number of folds: 5
    Number skipped folds: 0 
    Number of steps per fold: 14
    Number of steps to exclude between last observed data (last window) and predictions (gap): 0
    Last fold only includes 9 observations.

Fold: 0
    Training:   0 -- 299  (n=300)
    Validation: 300 -- 313  (n=14)
Fold: 1
    Training:   No training in this fold
    Validation: 314 -- 327  (n=14)
Fold: 2
    Training:   No training in this fold
    Validation: 328 -- 341  (n=14)
Fold: 3
    Training:   No training in this fold
    Validation: 342 -- 355  (n=14)
Fold: 4
    Training:   No training in this fold
    Validation: 356 -- 364  (n=9)



100%|██████████| 5/5 [00:00<00:00, 618.79it/s]

Auto-selected forecaster: ForecasterRecursive
Auto-selected estimator: LGBMRegressor
Uses exog: True

Explanation:
Using 82% of data (300 observations) for initial training, expanding window, no refit, 14-step horizon, 5 folds. Results — mean_absolute_error: 4.1101, mean_squared_error: 24.2056, mean_absolute_scaled_error: 1.4188, mean_absolute_percentage_error: 0.0427.

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,4.110126,24.205572,1.41884,0.042661


In [23]:
print(result_shortcut.code)

import pandas as pd
from lightgbm import LGBMRegressor
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import backtesting_forecaster, TimeSeriesFold
from skforecast.preprocessing import RollingFeatures

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'mean'],
    window_sizes = [7, 91],
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator            = LGBMRegressor(random_state=123, verbose=-1),
    lags                 = [1, 3, 7, 8, 10, 12, 118],
    window_features      = window_features,
    categorical_features = 'auto',
    dropna_from_series   = False,
)

# Cross-validation configuration
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = 300,
    refit              = False,
)

# Run backtesting
exog_features = ['promo', '